In [1]:
import os
import numpy as np
from scipy.io import loadmat
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import RMSprop
import tensorflow as tf
from tensorflow.keras.layers import Input, LocallyConnected2D, Activation, Conv2D, Flatten, Dense, concatenate, Lambda, BatchNormalization
from tensorflow.keras.initializers import GlorotUniform
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K

# 데이터 경로
adhd_path = r'C:\2024_2_CD\Siamese\Data_brainmapping\ADHD'
control_path = r'C:\2024_2_CD\Siamese\Data_brainmapping\Control'

# 데이터 로드
def load_mat_files(folder_path):
    data_list = []
    for file_name in os.listdir(folder_path):
        if file_name.endswith('.mat'):
            file_path = os.path.join(folder_path, file_name)
            mat_data = loadmat(file_path)
            key = "brainmaps"
            if key in mat_data:
                data_list.append(mat_data[key])
    return np.array(data_list)

# 쌍 생성
def create_pairs_from_data(adhd_data, control_data):
    pairs, labels = [], []

    # Positive pairs (ADHD - ADHD)
    for i in range(len(adhd_data)):
        for j in range(i + 1, len(adhd_data)):
            pairs.append([adhd_data[i], adhd_data[j]])
            labels.append(1)

    # Negative pairs (ADHD - Control)
    for adhd_sample in adhd_data:
        for control_sample in control_data:
            pairs.append([adhd_sample, control_sample])
            labels.append(0)

    return np.array(pairs), np.array(labels)

# 데이터 준비
def prepare_input(pairs):
    inputs = [[] for _ in range(5)]
    for pair in pairs:
        inputs[0].append(pair[:, :, :4])    # Delta
        inputs[1].append(pair[:, :, 4:8])   # Theta
        inputs[2].append(pair[:, :, 8:12])  # Alpha
        inputs[3].append(pair[:, :, 12:35]) # Beta
        inputs[4].append(pair[:, :, 35:])   # Gamma

    return [np.array(input_data) for input_data in inputs]

# 모델 정의
def my_model(seed=42):
    inputs = [
        Input(shape=(16, 16, 4)),
        Input(shape=(16, 16, 4)),
        Input(shape=(16, 16, 4)),
        Input(shape=(16, 16, 23)),
        Input(shape=(16, 16, 5))
    ]

    features = []
    for inp in inputs:
        x = LocallyConnected2D(1, (5, 5), kernel_initializer=GlorotUniform(seed=seed))(inp)
        x = Activation("hard_sigmoid")(x)
        features.append(x)

    x = concatenate(features, axis=-1)
    x = Conv2D(16, (5, 5), activation="tanh")(x)
    x = Conv2D(16, (5, 5), activation="tanh")(x)
    x = Flatten()(x)
    output = Dense(16, activation="tanh")(x)
    return Model(inputs=inputs, outputs=output)

def euclidean_distance(vects):
    x, y = vects
    sum_square = K.sum(K.square(x - y), axis=1, keepdims=True)
    return K.sqrt(K.maximum(sum_square, K.epsilon()))

# 샴 네트워크 설정
base_network = my_model()

input_a = [Input(shape=(16, 16, 4)), Input(shape=(16, 16, 4)), Input(shape=(16, 16, 4)), 
           Input(shape=(16, 16, 23)), Input(shape=(16, 16, 5))]
input_b = [Input(shape=(16, 16, 4)), Input(shape=(16, 16, 4)), Input(shape=(16, 16, 4)), 
           Input(shape=(16, 16, 23)), Input(shape=(16, 16, 5))]

feature_a = base_network(input_a)
feature_b = base_network(input_b)

# 거리 계산
distance = Lambda(euclidean_distance)([feature_a, feature_b])

# Batch Normalization과 활성화 함수(sigmoid) 추가
distance_normalized = BatchNormalization()(distance)
output = Activation("sigmoid")(distance_normalized)
siamese_model = Model(inputs=input_a + input_b, outputs=output)

# Contrastive Loss
def contrastive_loss_fn(margin=1):
    def contrastive_loss(y_true, y_pred):
        y_true = tf.cast(y_true, dtype=tf.float32)
        square_pred = tf.math.square(y_pred)
        margin_square = tf.math.square(tf.math.maximum(margin - y_pred, 0))
        return tf.math.reduce_mean(((1.0 - y_true) * square_pred) + y_true * margin_square)
    return contrastive_loss

optimizer = RMSprop(learning_rate=1e-4)
siamese_model.compile(optimizer=optimizer, loss=contrastive_loss_fn(margin=1), metrics=["accuracy"])


In [2]:
# ADHD와 Control 데이터 로드
adhd_data = load_mat_files(adhd_path)
control_data = load_mat_files(control_path)

# 데이터 확인
print(f"ADHD Data Shape: {adhd_data.shape}")
print(f"Control Data Shape: {control_data.shape}")

# ADHD와 Control 데이터를 각각 피험자 단위로 쌍 구성
adhd_subjects = [np.array(adhd_data[i]) for i in range(len(adhd_data))]
control_subjects = [np.array(control_data[i]) for i in range(len(control_data))]

# 피험자 단위로 Train-Test Split
adhd_train, adhd_test = train_test_split(adhd_subjects, test_size=0.2, random_state=42)
control_train, control_test = train_test_split(control_subjects, test_size=0.2, random_state=42)

# Train과 Test 쌍 생성
train_pairs, train_labels = create_pairs_from_data(np.array(adhd_train), np.array(control_train))
test_pairs, test_labels = create_pairs_from_data(np.array(adhd_test), np.array(control_test))

# Train 데이터를 다시 Split하여 Validation Set 구성
train_pairs, val_pairs, train_labels, val_labels = train_test_split(
    train_pairs, train_labels, test_size=0.1, random_state=42
)

print(f"Final Train Pairs: {train_pairs.shape}, Train Labels: {train_labels.shape}")
print(f"Validation Pairs: {val_pairs.shape}, Validation Labels: {val_labels.shape}")
print(f"Test Pairs: {test_pairs.shape}, Test Labels: {test_labels.shape}")

ADHD Data Shape: (61, 16, 16, 40)
Control Data Shape: (60, 16, 16, 40)
Final Train Pairs: (3088, 2, 16, 16, 40), Train Labels: (3088,)
Validation Pairs: (344, 2, 16, 16, 40), Validation Labels: (344,)
Test Pairs: (234, 2, 16, 16, 40), Test Labels: (234,)


In [3]:
# 데이터 준비
train_left = prepare_input(train_pairs[:, 0])
train_right = prepare_input(train_pairs[:, 1])
val_left = prepare_input(val_pairs[:, 0])
val_right = prepare_input(val_pairs[:, 1])
test_left = prepare_input(test_pairs[:, 0])
test_right = prepare_input(test_pairs[:, 1])

# 모델 학습
history = siamese_model.fit(
    train_left + train_right,
    train_labels,
    validation_data=(val_left + val_right, val_labels),
    batch_size=110,
    epochs=50,
)

# Train 데이터 평가
train_loss, train_accuracy = siamese_model.evaluate(train_left + train_right, train_labels)
print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}")

# Test 데이터 평가
test_loss, test_accuracy = siamese_model.evaluate(test_left + test_right, test_labels)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")


Epoch 1/50
29/29 [==============================] - 23s 332ms/step - loss: 0.2941 - accuracy: 0.5191 - val_loss: 0.2846 - val_accuracy: 0.3953
Epoch 2/50
29/29 [==============================] - 2s 75ms/step - loss: 0.2759 - accuracy: 0.5696 - val_loss: 0.2728 - val_accuracy: 0.3953
Epoch 3/50
29/29 [==============================] - 2s 72ms/step - loss: 0.2634 - accuracy: 0.6023 - val_loss: 0.2612 - val_accuracy: 0.4709
Epoch 4/50
29/29 [==============================] - 2s 70ms/step - loss: 0.2524 - accuracy: 0.6208 - val_loss: 0.2537 - val_accuracy: 0.5087
Epoch 5/50
29/29 [==============================] - 2s 72ms/step - loss: 0.2401 - accuracy: 0.6312 - val_loss: 0.2458 - val_accuracy: 0.5494
Epoch 6/50
29/29 [==============================] - 2s 71ms/step - loss: 0.2243 - accuracy: 0.6477 - val_loss: 0.2383 - val_accuracy: 0.6221
Epoch 7/50
29/29 [==============================] - 2s 71ms/step - loss: 0.2099 - accuracy: 0.6648 - val_loss: 0.2336 - val_accuracy: 0.6570
Epoch 8/50


In [4]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# 모델 예측
y_pred_probs = siamese_model.predict(test_left + test_right)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()  # Threshold 0.5 적용
y_pred_probs_pos = y_pred_probs.flatten()  # 확률 값 벡터화

# 평가 지표 계산
fold_accuracy = accuracy_score(test_labels, y_pred)
fold_precision_adhd = precision_score(test_labels, y_pred, pos_label=1)
fold_precision_control = precision_score(test_labels, y_pred, pos_label=0)
fold_recall_adhd = recall_score(test_labels, y_pred, pos_label=1)
fold_recall_control = recall_score(test_labels, y_pred, pos_label=0)
fold_f1_adhd = f1_score(test_labels, y_pred, pos_label=1)
fold_f1_control = f1_score(test_labels, y_pred, pos_label=0)
fold_auc = roc_auc_score(test_labels, y_pred_probs_pos)

# 결과 출력
print("\nEvaluation Metrics:")
print(f"Accuracy: {fold_accuracy:.4f}")
print(f"Precision ADHD: {fold_precision_adhd:.4f}")
print(f"Precision Control: {fold_precision_control:.4f}")
print(f"Recall ADHD: {fold_recall_adhd:.4f}")
print(f"Recall Control: {fold_recall_control:.4f}")
print(f"F1-score ADHD: {fold_f1_adhd:.4f}")
print(f"F1-score Control: {fold_f1_control:.4f}")
print(f"AUC: {fold_auc:.4f}")

8/8 [==============================] - 7s 18ms/step

Evaluation Metrics:
Accuracy: 0.5299
Precision ADHD: 0.3298
Precision Control: 0.6643
Recall ADHD: 0.3974
Recall Control: 0.5962
F1-score ADHD: 0.3605
F1-score Control: 0.6284
AUC: 0.4565
